In [ ]:
# ============================================================
# [셀 1] 프로젝트 개요
# - 이 노트북은 한국어 뉴스 기사 제목 분류 프로젝트를 위한 실습
# - 목표:
#   1) KLUE YNAT 데이터셋 로드
#   2) 데이터 확인 및 기댓값 테스트
#   3) TF-IDF + Logistic Regression 베이스라인 구축
#   4) BERT Fine-tuning
#   5) 암기 테스트 / 행동 테스트
# ============================================================

print("한국어 뉴스 기사 분류 프로젝트 노트북 시작")

In [1]:
# ============================================================
#  패키지 설치
# ============================================================

# !pip install -q datasets transformers evaluate scikit-learn pytorch-lightning torchmetrics

In [1]:
# ============================================================
#  환경 점검
# - 현재 Python, PyTorch, CUDA, GPU 정보를 확인합니다.
# - 사용자의 PC에서 BERT 미세조정이 가능한지 기본 점검하는 셀입니다.
# ============================================================

import sys
import platform

print("Python 버전:", sys.version)
print("운영체제:", platform.platform())

try:
    import torch
    print("PyTorch 버전:", torch.__version__)
    print("CUDA 사용 가능:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU 이름:", torch.cuda.get_device_name(0))
        total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU 메모리(GB): {total_mem:.1f}")
except Exception as e:
    print("PyTorch 확인 중 오류:", e)

Python 버전: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]
운영체제: Windows-10-10.0.19045-SP0
PyTorch 버전: 2.6.0+cu124
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3090
GPU 메모리(GB): 24.0


In [3]:
# ============================================================
# [셀 4] 데이터셋 로드
# - KLUE의 YNAT 뉴스 분류 데이터셋을 Hugging Face datasets로 불러옴
# - 이 데이터셋은 한국어 뉴스 제목을 7개 카테고리로 분류하는 문제
# ============================================================

from datasets import load_dataset

dataset = load_dataset("klue", "ynat")
label_names = dataset["train"].features["label"].names

print(dataset)
print("라벨 목록:", label_names)

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})
라벨 목록: ['IT과학', '경제', '사회', '생활문화', '세계', '스포츠', '정치']


In [4]:
# ============================================================
# 데이터 기본 확인
# - 학습/검증 데이터 수 확인
# - 각 split에서 몇 개의 샘플이 있는지 빠르게 점검
# ============================================================

print("학습 데이터 개수:", len(dataset["train"]))
print("검증 데이터 개수:", len(dataset["validation"]))

학습 데이터 개수: 45678
검증 데이터 개수: 9107


In [5]:
# ============================================================
# 샘플 데이터 확인
# - 실제 뉴스 제목과 라벨을 몇 개 확인
# ============================================================

for i in range(5):
    sample = dataset["train"][i]
    label_id = sample["label"]
    print(f"[{i}] 라벨: {label_names[label_id]}")
    print("제목:", sample["title"])
    print("-" * 60)

[0] 라벨: 생활문화
제목: 유튜브 내달 2일까지 크리에이터 지원 공간 운영
------------------------------------------------------------
[1] 라벨: 생활문화
제목: 어버이날 맑다가 흐려져…남부지방 옅은 황사
------------------------------------------------------------
[2] 라벨: 사회
제목: 내년부터 국가RD 평가 때 논문건수는 반영 않는다
------------------------------------------------------------
[3] 라벨: 사회
제목: 김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것
------------------------------------------------------------
[4] 라벨: 생활문화
제목: 회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간
------------------------------------------------------------


In [9]:
# ============================================================
# 클래스 분포 확인
# - 각 카테고리의 데이터 개수를 확인
# - 지나친 불균형이 있는지 점검
# ============================================================

from collections import Counter

train_label_counts = Counter(dataset["train"]["label"])

print("클래스 분포")
for label_id, count in sorted(train_label_counts.items()):
    ratio = count / len(dataset["train"])
    print(f"{label_names[label_id]:6s} | {count:5d} | {ratio:.2%}")

클래스 분포
IT과학   |  5235 | 11.46%
경제     |  6118 | 13.39%
사회     |  5133 | 11.24%
생활문화   |  5751 | 12.59%
세계     |  8320 | 18.21%
스포츠    |  7742 | 16.95%
정치     |  7379 | 16.15%


In [11]:
# ============================================================
# 제목 길이 통계 확인
# - 뉴스 제목의 길이 분포 확인
# - 토큰 길이, max_length 설정 전에 확인
# ============================================================

title_lengths = [len(title) for title in dataset["train"]["title"]]

print("제목 길이 최소:", min(title_lengths))
print("제목 길이 최대:", max(title_lengths))
print("제목 길이 평균:", round(sum(title_lengths) / len(title_lengths), 2))

제목 길이 최소: 4
제목 길이 최대: 44
제목 길이 평균: 27.37


In [13]:
# ============================================================
# 데이터 기댓값 테스트 함수 정의
# - FSDL Day 3의 핵심 중 하나인 데이터 expectation test를 작성
# - 데이터가 '말이 되는 상태'인지 자동 점검
# - 너무 엄격하지 않게, 느슨하지만 유용한 테스트로 구성
# ============================================================

from collections import Counter

def run_data_expectation_tests(ds):
    # 1. 데이터 개수 최소 기준
    assert len(ds["train"]) > 1000, "train 데이터가 너무 적습니다."
    assert len(ds["validation"]) > 100, "validation 데이터가 너무 적습니다."

    # 2. 빈 제목이 없어야 함
    for split in ["train", "validation"]:
        titles = ds[split]["title"]
        empty_count = sum(1 for t in titles if not t or t.strip() == "")
        assert empty_count == 0, f"{split}에 빈 제목이 존재합니다. empty_count={empty_count}"

    # 3. 라벨 범위가 0~6인지 확인
    for split in ["train", "validation"]:
        labels = ds[split]["label"]
        assert min(labels) >= 0, f"{split}의 최소 라벨이 0보다 작습니다."
        assert max(labels) <= 6, f"{split}의 최대 라벨이 6보다 큽니다."

    # 4. 심각한 클래스 불균형이 없는지 느슨하게 확인
    counts = Counter(ds["train"]["label"])
    total = len(ds["train"])
    for label_id, count in counts.items():
        ratio = count / total
        assert ratio > 0.03, f"class {label_id} 비율이 너무 낮습니다. ratio={ratio:.2%}"

    # 5. 제목 길이가 비정상적으로 짧거나 긴지 확인
    lengths = [len(t) for t in ds["train"]["title"]]
    assert min(lengths) >= 2, "너무 짧은 제목이 있습니다."
    assert max(lengths) <= 500, "비정상적으로 긴 제목이 있습니다."

    return "모든 데이터 기댓값 테스트를 통과했습니다."

In [15]:
# ============================================================
# 데이터 기댓값 테스트 실행
# ============================================================

print(run_data_expectation_tests(dataset))

모든 데이터 기댓값 테스트를 통과했습니다.


In [17]:
# ============================================================
# TF-IDF + Logistic Regression 베이스라인 준비
# - FSDL에서 강조하는 'Make it run, make it right' 흐름에 맞춤
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score

X_train_text = dataset["train"]["title"]
X_val_text = dataset["validation"]["title"]
y_train = dataset["train"]["label"]
y_val = dataset["validation"]["label"]

vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(X_train_text)
X_val = vectorizer.transform(X_val_text)

print("TF-IDF 변환 완료")
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

TF-IDF 변환 완료
X_train shape: (45678, 10000)
X_val shape: (9107, 10000)


In [19]:
# ============================================================
# TF-IDF 베이스라인 학습
# - Logistic Regression으로 간단한 텍스트 분류 모델을 학습
# - 이 결과가 이후 BERT 모델의 비교 기준선(baseline)이 됨
# ============================================================

# baseline_model = LogisticRegression(max_iter=1000, multi_class="multinomial")
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_val)

baseline_acc = accuracy_score(y_val, y_pred_baseline)
baseline_f1 = f1_score(y_val, y_pred_baseline, average="macro")

print("Baseline Accuracy:", round(baseline_acc, 4))
print("Baseline Macro F1:", round(baseline_f1, 4))

Baseline Accuracy: 0.6841
Baseline Macro F1: 0.6898


In [21]:
# ============================================================
# TF-IDF 베이스라인 상세 리포트
# - 클래스별 precision / recall / f1-score를 확인
# ============================================================

print(classification_report(y_val, y_pred_baseline, target_names=label_names))

              precision    recall  f1-score   support

        IT과학       0.63      0.72      0.67       554
          경제       0.71      0.73      0.72      1348
          사회       0.78      0.60      0.68      3701
        생활문화       0.67      0.71      0.69      1369
          세계       0.52      0.73      0.61       835
         스포츠       0.77      0.87      0.81       578
          정치       0.55      0.76      0.64       722

    accuracy                           0.68      9107
   macro avg       0.66      0.73      0.69      9107
weighted avg       0.70      0.68      0.69      9107



In [24]:
# ============================================================
# BERT 토크나이저 로드
# - 한국어 뉴스 분류를 위해 klue/bert-base 토크나이저를 불러옴
# - 제목 텍스트를 모델 입력용 토큰으로 변환할 준비
# ============================================================

from transformers import AutoTokenizer

MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("토크나이저 로드 완료:", MODEL_NAME)

토크나이저 로드 완료: klue/bert-base


In [25]:
# ============================================================
# 토큰화 함수 정의
# - 뉴스 제목을 BERT 입력 형식으로 변환
# - max_length=128은 제목 분류 문제에서는 충분함
# ============================================================

def tokenize_fn(examples):
    return tokenizer(
        examples["title"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

In [30]:
# ============================================================
# 데이터셋 토큰화
# - Hugging Face dataset 전체에 토큰화를 적용
# - token_type_ids까지 포함해서 모델 입력 형식을 맞춤
# ============================================================

tokenized_dataset = dataset.map(tokenize_fn, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'labels', 'url', 'date', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'labels', 'url', 'date', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9107
    })
})


In [32]:
# ============================================================
# DataModule 정의
# - PyTorch Lightning용 DataModule을 정의
# - Windows 환경에서는 num_workers=0으로 시작하는 것이 안전함
# ============================================================

import os
import pytorch_lightning as pl
from torch.utils.data import DataLoader

class NewsDataModule(pl.LightningDataModule):
    def __init__(self, tokenized_ds, batch_size=16, num_workers=None):
        super().__init__()
        self.tokenized_ds = tokenized_ds
        self.batch_size = batch_size

        if num_workers is not None:
            self.num_workers = num_workers
        elif os.name == "nt":
            self.num_workers = 0
        else:
            self.num_workers = min(4, os.cpu_count() or 1)

    def setup(self, stage=None):
        self.train_ds = self.tokenized_ds["train"]
        self.val_ds = self.tokenized_ds["validation"]

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

In [34]:
# ============================================================
# [셀 18] Lightning 모델 정의
# - klue/bert-base를 뉴스 분류용으로 미세조정하는 LightningModule
# - train_loss, val_loss, val_f1를 기록함
# - token_type_ids가 들어와도 처리할 수 있게 수정한 버전
# ============================================================

import torch
from transformers import AutoModelForSequenceClassification
from torchmetrics import F1Score
import pytorch_lightning as pl

class NewsClassifier(pl.LightningModule):
    def __init__(self, model_name=MODEL_NAME, num_labels=7, lr=2e-5):
        super().__init__()
        self.save_hyperparameters()

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels
        )

        self.train_f1 = F1Score(task="multiclass", num_classes=num_labels, average="macro")
        self.val_f1 = F1Score(task="multiclass", num_classes=num_labels, average="macro")

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )

    def training_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch.get("token_type_ids", None),
            labels=batch["labels"]
        )
        loss = outputs.loss
        preds = outputs.logits.argmax(dim=-1)

        self.train_f1(preds, batch["labels"])
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_f1", self.train_f1, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch.get("token_type_ids", None),
            labels=batch["labels"]
        )
        loss = outputs.loss
        preds = outputs.logits.argmax(dim=-1)

        self.val_f1(preds, batch["labels"])
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        self.log("val_f1", self.val_f1, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        return optimizer

In [36]:
# ============================================================
# DataModule / 모델 인스턴스 생성
# - 실제 학습에 사용할 데이터모듈과 모델 객체를 생성
# - 첫 실행은 batch_size=16 정도로 시작
# ============================================================

data_module = NewsDataModule(tokenized_dataset, batch_size=16)
model = NewsClassifier(lr=2e-5)

print("DataModule 및 모델 생성 완료")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

DataModule 및 모델 생성 완료


In [38]:
# ============================================================
# 학습 전 Shape / NaN 점검
# - PR 단계 테스트 중 일부
# - 배치 shape와 NaN 여부를 먼저 확인해 간단한 버그를 빨리 잡음
# ============================================================

data_module.setup()
train_loader = data_module.train_dataloader()
batch = next(iter(train_loader))

print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)

assert batch["input_ids"].ndim == 2
assert batch["attention_mask"].ndim == 2
assert batch["labels"].ndim == 1

assert not torch.isnan(batch["input_ids"].float()).any()
assert not torch.isnan(batch["attention_mask"].float()).any()

print("Shape / NaN 점검 통과")

input_ids shape: torch.Size([16, 128])
attention_mask shape: torch.Size([16, 128])
labels shape: torch.Size([16])
Shape / NaN 점검 통과


In [40]:
# ============================================================
# 본 학습 실행
# - GPU가 있으면 mixed precision을 사용함
# - CPU 환경이면 먼저 1 epoch + 일부 배치만으로 스모크 테스트를 권장
# ============================================================

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath="checkpoints",
    filename="best-{epoch}-{val_f1:.3f}",
    monitor="val_f1",
    mode="max",
    save_top_k=1,
    save_last=True
)

early_stop_callback = pl.callbacks.EarlyStopping(
    monitor="val_f1",
    patience=2,
    mode="max"
)

trainer = pl.Trainer(
    accelerator="auto",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else 32,
    max_epochs=3 if torch.cuda.is_available() else 1,
    limit_train_batches=1.0 if torch.cuda.is_available() else 0.1,
    gradient_clip_val=1.0,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=False
)

trainer.fit(model, datamodule=data_module)

Epoch 2/2  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2855/2855 0:02:45 • 0:00:00 17.29it/s train_loss_step: 0.427           
                                                                                  train_f1_step: 0.744 val_loss:   
                                                                                  0.619 val_f1: 0.852              
                                                                                  train_loss_epoch: 0.150          
                                                                                  train_f1_epoch: 0.949            

`Trainer.fit` stopped: `max_epochs=3` reached.


In [42]:
# ============================================================
# 검증 성능 확인
# - 저장된 최고 체크포인트 기준으로 검증 성능을 다시 확인
# - baseline(TF-IDF)보다 좋아졌는지 비교 포인트로 사용
# ============================================================

trainer.validate(model, datamodule=data_module, ckpt_path="best")

Validation ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570/570 0:00:10 • 0:00:00 52.93it/s

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val_f1           │    0.8621298670768738     │
│         val_loss          │    0.4273228347301483     │
└───────────────────────────┴───────────────────────────┘

[{'val_loss': 0.4273228347301483, 'val_f1': 0.8621298670768738}]

In [43]:
# ============================================================
# 암기 테스트 설명
# - 작은 배치를 일부러 과적합시켜서 모델/학습 파이프라인 자체가 정상인지 확인
# - 작은 데이터조차 못 외우면 데이터보다 구조나 학습 설정 문제일 가능성이 큽
# ============================================================

print("다음 셀에서 암기 테스트를 실행합니다.")

다음 셀에서 암기 테스트를 실행합니다.


In [44]:
# ============================================================
# 암기 테스트 실행
# - overfit_batches=1 설정으로 아주 작은 배치를 반복 학습
# - loss가 충분히 감소하면 기본 학습 경로는 정상으로 판단할 수 있음
# ============================================================

memorization_model = NewsClassifier(lr=1e-3)
memorization_trainer = pl.Trainer(
    overfit_batches=1,
    max_epochs=20,
    accelerator="auto",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else 32,
    logger=False,
    enable_checkpointing=False
)

memorization_trainer.fit(memorization_model, datamodule=data_module)
print("암기 테스트 완료")

Epoch 19/19 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:00:00 • 0:00:00 0.00it/s train_loss_step: 1.892              
                                                                               train_f1_step: 0.057 val_loss: 2.447
                                                                               val_f1: 0.000 train_loss_epoch:     
                                                                               1.892 train_f1_epoch: 0.057         

`Trainer.fit` stopped: `max_epochs=20` reached.


암기 테스트 완료


In [45]:
# ============================================================
# [셀 25] 최고 체크포인트 로드
# - 행동 테스트와 개별 예측을 위해 가장 좋은 체크포인트를 불러옴
# ============================================================

import os

ckpt_files = [f for f in os.listdir("checkpoints") if f.endswith(".ckpt")]
ckpt_files.sort()

best_ckpt_path = os.path.join("checkpoints", ckpt_files[-1])
print("불러올 체크포인트:", best_ckpt_path)

best_model = NewsClassifier.load_from_checkpoint(best_ckpt_path)
best_model.eval()
best_model.freeze()

device = "cuda" if torch.cuda.is_available() else "cpu"
best_model.to(device)

불러올 체크포인트: checkpoints\last.ckpt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

NewsClassifier(
  (model): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(32000, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768,

In [46]:
# ============================================================
# 단일 문장 예측 함수 정의
# - 임의의 뉴스 제목을 넣으면 카테고리를 예측하는 함수
# - 이후 행동 테스트와 데모에 사용함
# ============================================================

def predict_label(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = best_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_id = probs.argmax(dim=-1).item()
        confidence = probs.max(dim=-1).values.item()

    return {
        "text": text,
        "pred_id": pred_id,
        "pred_label": label_names[pred_id],
        "confidence": round(confidence, 4)
    }

In [47]:
# ============================================================
# 개별 예측 테스트
# - 사람이 보기에도 비교적 명확한 뉴스 제목을 넣어 예측 결과를 확인
# - 이것이 행동 테스트의 가장 간단한 시작점
# ============================================================

examples = [
    "삼성전자, 차세대 AI 반도체 공개",
    "코스피 장중 3000선 돌파",
    "대통령 국무회의서 민생 대책 논의",
    "손흥민 시즌 20호 골 기록",
]

for text in examples:
    result = predict_label(text)
    print(result)

{'text': '삼성전자, 차세대 AI 반도체 공개', 'pred_id': 0, 'pred_label': 'IT과학', 'confidence': 0.9359}
{'text': '코스피 장중 3000선 돌파', 'pred_id': 1, 'pred_label': '경제', 'confidence': 0.9846}
{'text': '대통령 국무회의서 민생 대책 논의', 'pred_id': 6, 'pred_label': '정치', 'confidence': 0.9861}
{'text': '손흥민 시즌 20호 골 기록', 'pred_id': 5, 'pred_label': '스포츠', 'confidence': 0.9953}


In [48]:
# ============================================================
# 행동 테스트 1: 명백한 사례 테스트
# - 사람이 봐도 거의 확실한 제목에서 모델이 납득 가능한 예측을 하는지 점검
# - 완벽 자동화 전의 간단한 품질 확인
# ============================================================

obvious_cases = [
    ("삼성전자, 새로운 AI 반도체 칩 발표", "IT과학"),
    ("코스피 장중 3000선 돌파 사상 최고", "경제"),
    ("대통령 국무회의 소집 민생 안정 논의", "정치"),
    ("손흥민 시즌 20호 골 프리미어리그 기록", "스포츠"),
]

for text, expected in obvious_cases:
    result = predict_label(text)
    print("입력:", text)
    print("예측:", result["pred_label"], "| 기대:", expected, "| confidence:", result["confidence"])
    print("-" * 60)

입력: 삼성전자, 새로운 AI 반도체 칩 발표
예측: IT과학 | 기대: IT과학 | confidence: 0.9277
------------------------------------------------------------
입력: 코스피 장중 3000선 돌파 사상 최고
예측: 경제 | 기대: 경제 | confidence: 0.984
------------------------------------------------------------
입력: 대통령 국무회의 소집 민생 안정 논의
예측: 정치 | 기대: 정치 | confidence: 0.986
------------------------------------------------------------
입력: 손흥민 시즌 20호 골 프리미어리그 기록
예측: 스포츠 | 기대: 스포츠 | confidence: 0.9942
------------------------------------------------------------


In [49]:
# ============================================================
# [셀 29] 행동 테스트 2: 표현이 조금 달라도 결과가 비슷한지 확인
# - 쉼표, 띄어쓰기, 표현 차이에도 결과가 크게 흔들리지 않는지 봅니다.
# - 이것은 간단한 불변성(invariance) 테스트 예시입니다.
# ============================================================

robust_pairs = [
    ("삼성전자 새 반도체 공장 착공", "삼성전자, 새 반도체 공장 착공"),
    ("미국 대선 후보 TV토론 개최", "미국 대선 후보 TV 토론 개최"),
    ("코스피 3000 돌파", "코스피 3000선 돌파"),
]

for a, b in robust_pairs:
    result_a = predict_label(a)
    result_b = predict_label(b)

    print("문장 A:", a, "->", result_a["pred_label"], result_a["confidence"])
    print("문장 B:", b, "->", result_b["pred_label"], result_b["confidence"])
    print("-" * 60)

문장 A: 삼성전자 새 반도체 공장 착공 -> 경제 0.9101
문장 B: 삼성전자, 새 반도체 공장 착공 -> 경제 0.9018
------------------------------------------------------------
문장 A: 미국 대선 후보 TV토론 개최 -> 세계 0.9747
문장 B: 미국 대선 후보 TV 토론 개최 -> 세계 0.9749
------------------------------------------------------------
문장 A: 코스피 3000 돌파 -> 경제 0.9787
문장 B: 코스피 3000선 돌파 -> 경제 0.9827
------------------------------------------------------------


In [50]:
# ============================================================
# [셀 30] Part A 정리
# - 첨부파일의 16.3 이하 내용 중 Part A를
#   '한국어 뉴스 기사 분류 프로젝트' 기준으로 채운 예시입니다.
# ============================================================

part_a = """
[Part A — 문제 정의와 전략]

프로젝트명:
한국어 뉴스 기사 제목 분류

문제 정의:
한국어 뉴스 기사 제목을 7개 카테고리로 자동 분류한다.

ML이 필요한 이유:
같은 단어라도 문맥에 따라 경제, 정치, IT 등으로 달라질 수 있어
규칙 기반 분류의 한계가 크다.

접근 방식:
- API 호출: 아니오
- Fine-tuning: 예
- 처음부터 학습: 아니오

아키타입:
Software 2.0

성공 메트릭:
- 비즈니스: 사람이 수동 분류하는 부담 감소
- ML: Macro F1 0.85 이상

PoC 목표:
1주 내에 TF-IDF 베이스라인보다 높은 성능의 BERT 분류기 구현
"""
print(part_a)


[Part A — 문제 정의와 전략]

프로젝트명:
한국어 뉴스 기사 제목 분류

문제 정의:
한국어 뉴스 기사 제목을 7개 카테고리로 자동 분류한다.

ML이 필요한 이유:
같은 단어라도 문맥에 따라 경제, 정치, IT 등으로 달라질 수 있어
규칙 기반 분류의 한계가 크다.

접근 방식:
- API 호출: 아니오
- Fine-tuning: 예
- 처음부터 학습: 아니오

아키타입:
Software 2.0

성공 메트릭:
- 비즈니스: 사람이 수동 분류하는 부담 감소
- ML: Macro F1 0.85 이상

PoC 목표:
1주 내에 TF-IDF 베이스라인보다 높은 성능의 BERT 분류기 구현



In [51]:
# ============================================================
# [셀 31] Part B 정리
# - 첨부파일의 16.3 이하 내용 중 Part B를
#   프로젝트 기준으로 채운 예시입니다.
# ============================================================

part_b = """
[Part B — 인프라와 데이터]

환경:
- conda 사용
- pip-tools로 의존성 고정 가능

프레임워크:
PyTorch Lightning + Transformers

GPU 계획:
RTX 3090 1대 사용 가능
(로컬 GPU로 충분, 필요 시 Colab 보조 가능)

예상 비용:
로컬 GPU 사용 시 사실상 0달러

데이터 확보:
KLUE YNAT 공개 데이터셋 사용

레이블링 전략:
기존 공개 라벨 활용

데이터 버전 관리:
현재 Level 1
목표 Level 2

실험 관리:
W&B 또는 최소한 수동 로그 기록
"""
print(part_b)


[Part B — 인프라와 데이터]

환경:
- conda 사용
- pip-tools로 의존성 고정 가능

프레임워크:
PyTorch Lightning + Transformers

GPU 계획:
RTX 3090 1대 사용 가능
(로컬 GPU로 충분, 필요 시 Colab 보조 가능)

예상 비용:
로컬 GPU 사용 시 사실상 0달러

데이터 확보:
KLUE YNAT 공개 데이터셋 사용

레이블링 전략:
기존 공개 라벨 활용

데이터 버전 관리:
현재 Level 1
목표 Level 2

실험 관리:
W&B 또는 최소한 수동 로그 기록



In [54]:
# ============================================================
# [셀 32] Part C 정리
# - 첨부파일의 16.3 이하 내용 중 Part C를
#   테스트와 품질 관점에서 채운 예시입니다.
# ============================================================

part_c = """
[Part C — 테스트와 품질]

PR 단계 테스트:
- Shape 테스트
- NaN 테스트
- 암기 테스트
- 린팅

야간 테스트:
- 전체 검증 성능 벤치마크
- 행동 테스트
- 데이터 기댓값 테스트

평가 전략:
- 자동 메트릭: Accuracy, Macro F1
- Human eval: 소규모 골든셋 확인 가능

드리프트 모니터링:
- 입력 드리프트
- 성능 드리프트

알림 채널:
- W&B
- 학습 로그
- GitHub Actions(선택)
"""
print(part_c)


[Part C — 테스트와 품질]

PR 단계 테스트:
- Shape 테스트
- NaN 테스트
- 암기 테스트
- 린팅

야간 테스트:
- 전체 검증 성능 벤치마크
- 행동 테스트
- 데이터 기댓값 테스트

평가 전략:
- 자동 메트릭: Accuracy, Macro F1
- Human eval: 소규모 골든셋 확인 가능

드리프트 모니터링:
- 입력 드리프트
- 성능 드리프트

알림 채널:
- W&B
- 학습 로그
- GitHub Actions(선택)



In [55]:
# ============================================================
# [셀 33] Part D 정리
# - 첨부파일의 16.3 이하 내용 중 Part D를
#   배포 연결 관점에서 채운 예시입니다.
# ============================================================

part_d = """
[Part D — 배포와 연결]

서빙 방식:
FastAPI

예상 API:
POST /classify

입력 예시:
{"title": "삼성전자 새 반도체 공장 착공"}

출력 예시:
{
  "category": "IT과학",
  "confidence": 0.95
}

체크포인트 로드:
PyTorch Lightning load_from_checkpoint 사용
"""
print(part_d)


[Part D — 배포와 연결]

서빙 방식:
FastAPI

예상 API:
POST /classify

입력 예시:
{"title": "삼성전자 새 반도체 공장 착공"}

출력 예시:
{
  "category": "IT과학",
  "confidence": 0.95
}

체크포인트 로드:
PyTorch Lightning load_from_checkpoint 사용



In [56]:
# ============================================================
# [셀 34] ML 실행 계획서 출력
# - 사용자가 제시한 템플릿을 뉴스 분류 프로젝트 기준으로 채운 예시입니다.
# - 과제 제출 전 초안으로 활용할 수 있습니다.
# ============================================================

project_plan = """
═══════════════════════════════════════════════════
       [프로젝트명] ML 실행 계획서
═══════════════════════════════════════════════════

■ Day 1 영역 — 전략과 기획
  문제 정의: 한국어 뉴스 기사 제목을 7개 카테고리로 자동 분류
  ML이 필요한 이유: 같은 단어도 문맥에 따라 다른 카테고리로 해석되어 규칙 기반 한계가 큼
  접근법: [ ] API 호출  [✓] Fine-tuning  [ ] 처음부터 학습
  아키타입: [✓] Software 2.0  [ ] Human-in-the-loop  [ ] Autonomous
  성공 메트릭: (비즈니스) 사람 수준 분류 품질 근접 / (ML) Macro F1 0.85 이상
  PoC 목표: 1주 내에 TF-IDF 베이스라인을 넘는 모델 달성

■ Day 2 영역 — 인프라와 데이터
  환경: [ ] uv  [✓] conda  / 의존성 잠금: [ ] uv lock  [✓] pip-tools
  프레임워크: PyTorch Lightning + Transformers
  GPU 계획: RTX 3090 × 1대 (클라우드: 필요 시 Colab)
  예상 비용: $0/실험 (스팟 인스턴스: [ ] 가능  [✓] 불가능)
  데이터 확보: KLUE YNAT 공개 데이터셋 활용
  레이블링 전략: [ ] Self-supervised  [ ] 합성  [✓] 수동 (업체: 없음, 공개 라벨 사용)
  데이터 버전 관리: Level 1
  실험 관리: [✓] W&B  [ ] MLflow  [ ] 기타: 수동 기록 보조

■ Day 3 영역 — 테스트와 품질
  PR 단계 테스트 (10분): [✓] Shape  [✓] NaN  [✓] 암기  [✓] 린팅
  야간 테스트: [✓] 성능 벤치마크  [✓] 행동 테스트  [✓] 데이터 기댓값
  평가 전략: [✓] 자동 메트릭  [ ] LLM-as-a-Judge  [ ] Human eval
  드리프트 모니터링: [✓] 입력  [ ] 출력  [✓] 성능
  알림 채널: W&B + 학습 로그

■ 리스크와 완화 계획
  가장 큰 리스크: 경제/IT/사회처럼 경계가 애매한 클래스 혼동
  완화 방법: 행동 테스트, Macro F1 중심 평가, 하이퍼파라미터/모델 비교
  되돌아갈 수 있는 시점: BERT가 baseline을 못 이기면 TF-IDF baseline으로 회귀
"""
print(project_plan)


═══════════════════════════════════════════════════
       [프로젝트명] ML 실행 계획서
═══════════════════════════════════════════════════

■ Day 1 영역 — 전략과 기획
  문제 정의: 한국어 뉴스 기사 제목을 7개 카테고리로 자동 분류
  ML이 필요한 이유: 같은 단어도 문맥에 따라 다른 카테고리로 해석되어 규칙 기반 한계가 큼
  접근법: [ ] API 호출  [✓] Fine-tuning  [ ] 처음부터 학습
  아키타입: [✓] Software 2.0  [ ] Human-in-the-loop  [ ] Autonomous
  성공 메트릭: (비즈니스) 사람 수준 분류 품질 근접 / (ML) Macro F1 0.85 이상
  PoC 목표: 1주 내에 TF-IDF 베이스라인을 넘는 모델 달성

■ Day 2 영역 — 인프라와 데이터
  환경: [ ] uv  [✓] conda  / 의존성 잠금: [ ] uv lock  [✓] pip-tools
  프레임워크: PyTorch Lightning + Transformers
  GPU 계획: RTX 3090 × 1대 (클라우드: 필요 시 Colab)
  예상 비용: $0/실험 (스팟 인스턴스: [ ] 가능  [✓] 불가능)
  데이터 확보: KLUE YNAT 공개 데이터셋 활용
  레이블링 전략: [ ] Self-supervised  [ ] 합성  [✓] 수동 (업체: 없음, 공개 라벨 사용)
  데이터 버전 관리: Level 1
  실험 관리: [✓] W&B  [ ] MLflow  [ ] 기타: 수동 기록 보조

■ Day 3 영역 — 테스트와 품질
  PR 단계 테스트 (10분): [✓] Shape  [✓] NaN  [✓] 암기  [✓] 린팅
  야간 테스트: [✓] 성능 벤치마크  [✓] 행동 테스트  [✓] 데이터 기댓값
  평가 전략: [✓] 자동 메트릭  [ ] LLM-as-a-Judge  [ ] Human